# RQ1: Baseline YOLOv8n Performance on Newcastle CCTV
**Author:** Sumit Malviya (W24041293) | **Module:** KF7029 | **Supervisor:** Dr. Jason Moore

**Purpose:** Establish baseline performance of COCO pre-trained YOLOv8n on the 344-image Newcastle test set.

**Key result:** 12.9% mAP@0.5 — confirming a 24.4pp domain gap vs COCO benchmark (37.3%)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

ROOT        = Path('..')
RESULTS_DIR = ROOT / 'data/results'
FIG_OUT     = Path('figures')
FIG_OUT.mkdir(exist_ok=True)

# ── Final confirmed baseline results ──────────────────────────────────────
# Source: Ultralytics .val() on 344-image test set, imgsz=960, conf=0.25
# NOTE: COCO baseline cannot be re-evaluated with v6_merged data.yaml due to
# class ID mismatch (COCO uses different class indices than the fine-tuned model).
# These results were produced by running the baseline against the original
# ground truth labels using the COCO class mapping.

BASELINE = {
    'mAP50':     12.9,
    'mAP50_95':   9.5,
    'Precision': 70.7,
    'Recall':    17.9,
    'F1':        28.5,
}

BASELINE_PERCLASS = {
    'car':           26.1,
    'large_vehicle': 11.7,
    'traffic_light':  9.1,
    'cyclist':         0.0,
    'person':         15.8,
}

print('='*60)
print('  RQ1: YOLOv8n COCO BASELINE — 344 TEST IMAGES')
print('='*60)
for k, v in BASELINE.items():
    print(f'  {k:<15s}: {v:.1f}%')
print()
print('  Per-class AP@0.5:')
for cls, ap in BASELINE_PERCLASS.items():
    bar = '█' * int(ap / 3)
    print(f'    {cls:<15s}: {ap:5.1f}%  {bar}')
print('='*60)
print()
print('  KEY FINDING:')
coco_bench = 37.3
gap = coco_bench - BASELINE["mAP50"]
print(f'  COCO benchmark mAP: {coco_bench}%')
print(f'  Newcastle CCTV mAP: {BASELINE["mAP50"]}%')
print(f'  Domain gap:         −{gap:.1f}pp')
print(f'  → Confirms need for domain-specific fine-tuning (RQ2)')

  RQ1: YOLOv8n COCO BASELINE — 344 TEST IMAGES
  mAP50          : 12.9%
  mAP50_95       : 9.5%
  Precision      : 70.7%
  Recall         : 17.9%
  F1             : 28.5%

  Per-class AP@0.5:
    car            :  26.1%  ████████
    large_vehicle  :  11.7%  ███
    traffic_light  :   9.1%  ███
    cyclist        :   0.0%  
    person         :  15.8%  █████

  KEY FINDING:
  COCO benchmark mAP: 37.3%
  Newcastle CCTV mAP: 12.9%
  Domain gap:         −24.4pp
  → Confirms need for domain-specific fine-tuning (RQ2)


## 2. Overall Metrics Chart

In [2]:
BLUE  = '#185FA5'; CORAL = '#C9432F'; AMBER = '#B45309'
LIGHT = '#C8C5C0'; GREEN = '#2E7D32'; MID   = '#555250'

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle(
    f'RQ1: YOLOv8n COCO Baseline — 344 test images\n'
    f'mAP@0.5 = {BASELINE["mAP50"]:.1f}% | Domain gap = −{37.3 - BASELINE["mAP50"]:.1f}pp vs COCO benchmark',
    fontsize=10, fontweight='bold'
)

# Panel A — overall metrics
ax = axes[0]
metrics = ['Precision', 'Recall', 'F1', 'mAP@0.5']
vals    = [BASELINE['Precision'], BASELINE['Recall'], BASELINE['F1'], BASELINE['mAP50']]
colors  = [BLUE, CORAL, AMBER, LIGHT]
bars = ax.bar(metrics, vals, color=colors, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+1,
            f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax.set_ylim(0, 85); ax.set_ylabel('%')
ax.set_title('Overall metrics — high precision but very low recall\n'
             'model detects very little on operational CCTV footage', fontsize=9)
ax.axhline(50, color='gray', lw=0.5, ls='--', alpha=0.4)

# Panel B — per-class AP
ax = axes[1]
classes  = list(BASELINE_PERCLASS.keys())
ap_vals  = list(BASELINE_PERCLASS.values())
b_colors = [BLUE if v > 5 else '#D0D0D0' for v in ap_vals]
bars2 = ax.bar(classes, ap_vals, color=b_colors, width=0.55)
ax.set_ylabel('AP@0.5 (%)')
ax.set_title('Per-class AP — all classes severely degraded\nvs COCO benchmark performance', fontsize=9)
ax.set_ylim(0, 40)
ax.tick_params(axis='x', rotation=15)
for bar, v in zip(bars2, ap_vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.3,
            f'{v:.1f}', ha='center', fontsize=8)
# Note cyclist = 0 (not in COCO)
ax.text(2.0, 1.5, 'not in COCO', ha='center', fontsize=6.5, color='gray')

plt.tight_layout()
fig.savefig(FIG_OUT / 'fig_rq1_baseline_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig_rq1_baseline_results.png')

Saved: figures/fig_rq1_baseline_results.png


/var/folders/03/4v9ll7hn78jg8p22yk35l3sm0000gn/T/ipykernel_58755/3994764479.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Domain Gap Visualisation

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Domain Gap: COCO Pre-trained YOLOv8n Performance Degradation',
             fontsize=11, fontweight='bold')

# Panel A — headline mAP comparison
ax = axes[0]
contexts   = ['COCO val\n(benchmark)', 'Newcastle CCTV\n(operational)']
maps_ctx   = [37.3, BASELINE['mAP50']]
colors_ctx = ['#93B8DD', '#C9432F']
bars = ax.bar(contexts, maps_ctx, color=colors_ctx, width=0.4)
for bar, v in zip(bars, maps_ctx):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.5,
            f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('mAP@0.5 (%)'); ax.set_ylim(0, 48)
ax.set_title(f'−{37.3-BASELINE["mAP50"]:.1f}pp domain gap', fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Annotation arrow
ax.annotate('', xy=(1, BASELINE['mAP50']+1), xytext=(0, 37.3-1),
            arrowprops=dict(arrowstyle='->', color=AMBER, lw=2))
ax.text(0.5, (37.3+BASELINE['mAP50'])/2,
        f'−{37.3-BASELINE["mAP50"]:.1f}pp', ha='center',
        fontsize=10, color=AMBER, fontweight='bold')

# Panel B — 4-way model progression (baseline → best)
ax = axes[1]
model_labels = ['YOLOv8n\nCOCO\n(baseline)', 'YOLOv8s\nCOCO',
                'YOLOv8s FT\n8-class', 'YOLOv8s FT\n5-class\n(final)']
maps_prog = [12.9, 16.3, 34.2, 65.1]
prog_colors = ['#C9432F', '#C8C5C0', '#93B8DD', '#185FA5']
bars2 = ax.bar(model_labels, maps_prog, color=prog_colors, width=0.55)
for bar, v in zip(bars2, maps_prog):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.8,
            f'{v}%', ha='center', fontsize=8, fontweight='bold')
ax.set_ylabel('mAP@0.5 (%)'); ax.set_ylim(0, 78)
ax.set_title('Progressive improvement through\ndomain adaptation (RQ1 → RQ2)', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIG_OUT / 'fig_rq1_domain_gap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig_rq1_domain_gap.png')

Saved: figures/fig_rq1_domain_gap.png


/var/folders/03/4v9ll7hn78jg8p22yk35l3sm0000gn/T/ipykernel_58755/3033839618.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Save Results

In [4]:
import csv

# Save overall
with open(RESULTS_DIR / 'rq1_baseline_overall.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    for k, v in BASELINE.items():
        w.writerow([k, v])

# Save per-class
with open(RESULTS_DIR / 'rq1_baseline_perclass.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['class', 'AP50'])
    for cls, ap in BASELINE_PERCLASS.items():
        w.writerow([cls, ap])

print('Results saved to data/results/')
print()
print('='*55)
print('  RQ1 SUMMARY FOR PAPER:')
print(f'  Baseline mAP@0.5    = {BASELINE["mAP50"]:.1f}%')
print(f'  Domain gap vs COCO  = −{37.3-BASELINE["mAP50"]:.1f}pp')
print(f'  Car AP              = {BASELINE_PERCLASS["car"]:.1f}%')
print(f'  Cyclist AP          = {BASELINE_PERCLASS["cyclist"]:.1f}% (not in COCO)')
print('  → Motivates RQ2 domain-specific fine-tuning')
print('='*55)

Results saved to data/results/

  RQ1 SUMMARY FOR PAPER:
  Baseline mAP@0.5    = 12.9%
  Domain gap vs COCO  = −24.4pp
  Car AP              = 26.1%
  Cyclist AP          = 0.0% (not in COCO)
  → Motivates RQ2 domain-specific fine-tuning


## RQ1 Complete ✅

**Confirmed result:** 12.9% mAP@0.5 on 344-image Newcastle test set

**Domain gap:** −24.4pp vs COCO benchmark (37.3%)

**Note on evaluation:** The COCO baseline cannot be directly re-evaluated using the `v6_merged/data.yaml` file because the COCO model predicts class IDs based on the 80-class COCO taxonomy, which does not align with the 5-class deployment taxonomy. The 12.9% result was established during the original four-way comparison using matched COCO class labels.

Proceed to → **RQ2_Finetune_Comparison_v2_updated.ipynb**